The main question is: How do we know if our RAG / Agentic RAG answer is good?

In ML, you used: accuracy, precision, recall, F1, ROC-AUC

In GenAI, we evaluate: 
- retrieval quality
- answer relevance
- groundedness
- hallucination risk

Concept Architecture:

Question
   ↓
Agentic RAG / RAG
   ↓
Retrieved Context
   ↓
Generated Answer
   ↓
Evaluation

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)

VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"
INDEX_NAME = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"
EMBEDDING_MODEL = "databricks-gte-large-en"
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"



In [0]:
#check endpoint exists or not
VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"

try:
    endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print("Endpoint already exists")

except Exception:
    print("Creating endpoint...")

    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )

In [0]:
#create index
SOURCE_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"
INDEX_NAME =  "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index_1"

embedding_dimension = len(
    spark.table(SOURCE_TABLE)
         .select("embedding")
         .first()["embedding"]
)

vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="customer_id",
    embedding_dimension=embedding_dimension,
    embedding_vector_column="embedding",
    columns_to_sync=[
        "customer_id",
        "note"
    ]
)

In [0]:
#check the state of the index
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)

print(index.describe())

In [0]:
def call_llm(prompt, max_tokens=300):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=max_tokens,
        temperature=0.0
    )

    return response.choices[0].message.content

In [0]:
def agentic_rag_agent(question):
    tool_name = choose_tool(question)

    print(f"Tool selected by LLM: {tool_name}")

    if "sql_count_notes" in tool_name:
        result = count_customer_notes()

        final_prompt = f"""
Question:
{question}

Tool used:
sql_count_notes

Tool result:
There are {result} customer notes.

Provide a clear final answer.
"""

    elif "vector_search_notes" in tool_name:
        context, rows = search_customer_notes(question)
        print("Executing Vector Search tool")
        final_prompt = f"""
You are a helpful telco customer support analyst.

Answer the user's question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    else:
        return f"Unable to choose a valid tool. LLM returned: {tool_name}"

    final_answer = call_llm(final_prompt)

    return final_answer

In [0]:
def count_customer_notes():
    count = spark.sql("""
        SELECT COUNT(*)
        FROM dbw_agentic_ai_dev.telco_ai.customer_notes
    """).collect()[0][0]

    return count

In [0]:
def choose_tool(question):
    tool_prompt = f"""
You are an agent. Choose the best tool for the user question.

Available tools:

1. sql_count_notes
Use this tool when the question asks for counts, totals, how many, averages, or table summaries.

2. vector_search_notes
Use this tool when the question asks why, reasons, complaints, dissatisfaction, cancellation, or customer sentiment.

Question:
{question}

Return only one tool name:
sql_count_notes
or
vector_search_notes
"""

    tool_name = call_llm(tool_prompt, max_tokens=20).strip().lower()

    return tool_name

In [0]:
def search_customer_notes(question, num_results=3):
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = [
        float(x)
        for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = results["result"]["data_array"]

    context_lines = []

    for customer_id, note, score in rows:
        context_lines.append(
            f"Customer {int(customer_id)}: {note}"
        )

    context = "\n".join(context_lines)

    return context, rows

In [0]:
#test set
eval_questions = [
    {
        "question": "Why are customers likely to cancel service?",
        "expected_keywords": ["cancel", "terminate", "service quality", "unhappy"]
    },
    {
        "question": "How many customer notes are there?",
        "expected_keywords": ["8"]
    }
]

In [0]:
#basic keyword evaluation
def keyword_score(answer, expected_keywords):
    answer_lower = answer.lower()
    matches = 0

    for keyword in expected_keywords:
        if keyword.lower() in answer_lower:
            matches += 1

    return matches / len(expected_keywords)

In [0]:
#LLM-as-Judge Evaluation
def judge_answer(question, answer):
    judge_prompt = f"""
You are evaluating an AI assistant answer.

Question:
{question}

Answer:
{answer}

Score the answer from 1 to 5 for:
1. Relevance
2. Groundedness
3. Clarity

Return only this format:
Relevance: <score>
Groundedness: <score>
Clarity: <score>
Reason: <short reason>
"""

    return call_llm(judge_prompt)


ML Evaluation:
prediction vs actual label

GenAI Evaluation:
question + context + answer quality

Relevance = Did it answer the question?
Groundedness = Did it stay within retrieved context?
Clarity = Is the response understandable?


In [0]:
#run agent
eval_results = []

for item in eval_questions:
    question = item["question"]
    answer = agentic_rag_agent(question)

    eval_results.append({
        "question": question,
        "answer": answer,
        "expected_keywords": item["expected_keywords"]
    })

display(spark.createDataFrame(eval_results))

scored_results = []

for row in eval_results:
    score = keyword_score(
        row["answer"],
        row["expected_keywords"]
    )

    scored_results.append({
        "question": row["question"],
        "answer": row["answer"],
        "keyword_score": score
    })

display(spark.createDataFrame(scored_results))

for row in eval_results:
    print("QUESTION:", row["question"])
    print(judge_answer(row["question"], row["answer"]))
    print("-" * 80)